# Environment

In [ ]:
# =========================
# Standard library
# =========================
import os
import re
from pathlib import Path
from typing import List, Tuple, Dict

# =========================
# Third-party libraries
# =========================
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# =========================
# Project-specific modules
# =========================
from kbase.config import settings
import kbase.preprocessing as dp


# Main Vectors

In [ ]:
key_col = "demandpk"
id_columns = ["healthcard_id", "national_id", "full_name"]

# Data Loading

## EDATOSCASO-LLAMADAS

In [ ]:
# Loading exported table
df_triaje = pq.read_table(os.path.join(settings.source_tables_path, settings.master_table_4demands_path)).to_pandas()
df_triaje.shape

In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(10)


# Run Pipeline in subet

In [ ]:
# Flag to test the pipeline in a subset (change subset in kbase/preprocessing.py)
## must be either "all", to run in all dataframe, or "test", to run it in a subset
df_triaje = dp.df_pipeline_test(df_triaje).sort_values(by="DEMANDAPK")
n_rows_source_data = len(df_triaje)

# Column Reduction


In [ ]:
# IDTIPODEMANDA1 and IDTIPODEMANDA2: additional information outside of the scope

# Columns to drop
columns_to_drop = ['IDTIPODEMANDA2', 'IDTIPODEMANDA3'] 

# Drop columns
df_triaje = df_triaje.drop(columns = columns_to_drop)
# df_triaje.info()

# Data cleaning

## Rename columns

In [ ]:
df_triaje = dp.rename_columns(df_triaje)


In [ ]:
dp.summarize_numbers(df_triaje, key_col, id_columns)


In [ ]:
df_triaje.head(1)

## DEMANDAPK - demandpk

In [ ]:
# Step 1: Count how many unique patients are linked to each demandpk
demand_patient_counts = (
    df_triaje.groupby('demandpk')['patientid']
    .nunique()
    .reset_index(name='n_patients')
)

# Step 2: Count distribution (optional)
distribution = (
    demand_patient_counts['n_patients']
    .value_counts(dropna=False)
    .sort_index()
)
print(distribution)

# ===============================================================
# Step 3: Identify demandpk with >= 2 linked patients
# ===============================================================
demandpk_mult = demand_patient_counts.loc[
    demand_patient_counts['n_patients'] >= 2, 
    'demandpk'
]

print(f"\nDemandPK with >=2 patients: {len(demandpk_mult)}")

# ===============================================================
# Step 4: Remove rows belonging to those demandpk
# ===============================================================
n_rows_before_demandpk_exclusion = len(df_triaje)

df_triaje = df_triaje[~df_triaje['demandpk'].isin(demandpk_mult)].reset_index(drop=True)

n_rows_after_demandpk_exclusion = len(df_triaje)

# Number of rows eliminated
n_rows_excluded_demandpk = n_rows_before_demandpk_exclusion - n_rows_after_demandpk_exclusion

print(f"Rows excluded due to duplicate demandpk (>=2 patients): {n_rows_excluded_demandpk}")
print(f"Rows after excluding duplicate demandpk: {n_rows_after_demandpk_exclusion}")

dp.summarize_numbers(df_triaje, key_col, id_columns)


## FECHADEMANDA - demand_date

In [ ]:
# Convert the raw string date
df_triaje["demand_date"] = pd.to_datetime(
    df_triaje["demand_date"],
    format="%Y%m%d",
    errors="coerce"
)

# Count how many failed conversions 
print(f'Number of invalid "demand_date" rows: {df_triaje["demand_date"].isna().sum()}')

df_triaje["demand_date"].dtype



## FNACIMIENTO - birth_date

In [ ]:
# Convert the raw string date
df_triaje["birth_date"] = pd.to_datetime(
    df_triaje["birth_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Count how many failed conversions 
print(f'Number of invalid "birth_date" rows: {df_triaje["birth_date"].isna().sum()}')

df_triaje["birth_date"].dtype

## EDAD - age

In [ ]:
df_triaje["age_type"].value_counts(dropna = False)

In [ ]:
df_triaje["age"].value_counts(dropna=False).sort_index(ascending=True)

In [ ]:
mask = (
    df_triaje["birth_date"].isna() &             # birth_date es NaT
    df_triaje["age"].notna() &              # call_age NO es NaN
    (df_triaje["age"] != 0)                 # call_age NO es 0
)

mask.sum()



In [ ]:

def process_age_pipeline(df, birth_col="birth_date", demand_col="demand_date",
                         age_col="age", age_type_col="age_type"):
    """
    Full age processing pipeline applied directly on `age` column.

    Steps:
    1. Compute age from birth_date when available.
    2. Compute age from reported age + age_type (A/M/D/S) when birth_date is missing.
    3. Replace the original 'age' column with the best available estimate.
    4. Round age to nearest integer.
    5. Convert final age to Int8.
    6. Remove auxiliary processing columns.

    Returns:
        The dataframe with an updated 'age' column (Int8).
    """

    # ------------------------------------------------------------
    # Step 1 — Compute age in years using birth_date
    # ------------------------------------------------------------
    df["age_from_birthdate"] = (
        (df[demand_col] - df[birth_col])
        .dt.total_seconds() / (365.25 * 24 * 3600)
    )

    # ------------------------------------------------------------
    # Step 2 — Define conversion multipliers for age_type
    # ------------------------------------------------------------
    unit_to_years = {
        "A": 1.0,        # years
        "M": 1/12,       # months
        "D": 1/365.25,   # days
        "S": 1/52,       # weeks
    }

    # ------------------------------------------------------------
    # Step 3 — Normalize age_type into uppercase strings
    # ------------------------------------------------------------
    df[age_type_col] = df[age_type_col].astype("string").str.upper()

    # ------------------------------------------------------------
    # Step 4 — Compute reported age in years using age and age_type
    # ------------------------------------------------------------
    df["reported_age_years"] = df[age_col] * df[age_type_col].map(unit_to_years)

    # ------------------------------------------------------------
    # Step 5 — Mask for rows missing birth_date but having reported age
    # ------------------------------------------------------------
    mask_missing_birth = (
        df[birth_col].isna() &
        df[age_col].notna() &
        (df[age_col] != 0)
    )

    # ------------------------------------------------------------
    # Step 6 — Update age directly
    # ------------------------------------------------------------
    # Start with age_from_birthdate when available
    df[age_col] = df["age_from_birthdate"]

    # Replace missing birthdate ages with reported_age_years
    df.loc[mask_missing_birth, age_col] = df.loc[mask_missing_birth, "reported_age_years"]

    # ------------------------------------------------------------
    # Step 7 — Round final age to nearest integer
    # ------------------------------------------------------------
    df[age_col] = df[age_col].round()

    # ------------------------------------------------------------
    # Step 8 — Convert to Int16
    # ------------------------------------------------------------
    df[age_col] = df[age_col].astype("Int16")

    # ------------------------------------------------------------
    # Step 9 — Remove auxiliary columns
    # ------------------------------------------------------------
    df = df.drop(columns=["age_from_birthdate", "reported_age_years"])

    return df

df_triaje = process_age_pipeline(df_triaje)


In [ ]:
mask = (
    df_triaje["birth_date"].isna() &             # birth_date es NaT
    df_triaje["age"].notna() &              # call_age NO es NaN
    (df_triaje["age"] != 0)                 # call_age NO es 0
)

mask.sum()

In [ ]:
## Exclusion of outliers and filtering by age range of interest
n_rows_before_age_exclusion = len(df_triaje)
print(f"Number of rows before age filtering: {n_rows_before_age_exclusion}")

df_triaje = df_triaje[df_triaje["age"].between(0, 105)]

n_rows_after_age_exclusion = len(df_triaje)

n_rows_excluded_age = n_rows_before_age_exclusion - n_rows_after_age_exclusion
print(f"Number of rows after age filtering: {n_rows_after_age_exclusion}")

print("Rows removed after excluding outliers and filtering by age range:",
      n_rows_excluded_age)


In [ ]:
# df_triaje_bebe = df_triaje[df_triaje["age"]==0]

In [ ]:
# df_triaje_bebe.head(100)

In [ ]:
df_triaje["age"].value_counts(dropna=False).sort_index().reset_index()


In [ ]:
df_triaje['age'].hist(bins=20, range=(0, 100), color='skyblue', edgecolor='black')
plt.title('Histogram of Age')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6) 
plt.show()

df_triaje.boxplot(column=['age'], by='demand_type_1')
plt.title('Boxplot of Age')
plt.ylabel('Age')
plt.show()


## SEX - sex

In [ ]:
print("Value counts of 'sex' column before conversion:")
print(df_triaje['sex'].value_counts(dropna=False))

In [ ]:
# Map dictionary
## 'D' and None values are unknown
## Male (H): 0
## Female (M): 1
map_sex = {
    'D': np.nan,
    'None': np.nan,
    'H': 0,
    'M': 1
}

# Map the values
df_triaje["sex"] = df_triaje["sex"].map(map_sex)

# Convert data type
df_triaje["sex"] = pd.to_numeric(df_triaje["sex"], errors="coerce")
df_triaje["sex"] = df_triaje["sex"].astype("Int64")
df_triaje["sex"] = df_triaje["sex"].astype("category")

# Check the mapping
print("\nValue counts of 'sex' column after data conversion:")
df_triaje["sex"].value_counts(dropna=False)


In [ ]:
## Exclusion of None
n_rows_before_sex_exclusion = len(df_triaje)
print(f"Number of rows before filtering out missing values in sex: {n_rows_before_sex_exclusion}")

# Filter out rows where sex is NaN
df_triaje = df_triaje[df_triaje["sex"].notna()]

n_rows_after_sex_exclusion = len(df_triaje)
print(f"Number of rows after filtering out missing values in sex: {n_rows_after_sex_exclusion}")

n_rows_excluded_sex = n_rows_before_sex_exclusion - n_rows_after_sex_exclusion

print(
    "Rows removed after filtering out missing values in sex:",
    n_rows_excluded_sex
)


In [ ]:
print("\nValue counts of 'sex' column after removing missing values:")
df_triaje["sex"].value_counts(dropna=False)


## IDASISTENCIA

In [ ]:
df_triaje["assistance_id"].value_counts(dropna=False)

## Time columns

In [ ]:
# List of time columns
time_columns = [
    "contact_entry_time",
    "contact_answer_time",
    "activation_time",
    "departure_time",
    "arrival_time",
    "load_time",
    "operative_time",
    "destination_time",
    "available_time"
]

for col in time_columns:
    df_triaje[col] = pd.to_datetime(
        df_triaje[col],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )

# Count NaT values after conversion
print("\nCount of invalid or missing datetime values after conversion:")
for col in time_columns:
    nat_count = df_triaje[col].isna().sum()
    dtype = df_triaje[col].dtype
    print(f"{col} ({dtype}): {nat_count} NaT values")


## IDPROVINCIA - province

In [ ]:
# Original value counts
print("Value counts of 'province' column before conversion:")
df_triaje["province"].value_counts(dropna=False)

In [ ]:
# Map dictionary
map_province = {
    4 : "ALM",
    11: "CAD",
    14: "COR",
    18: "GRA",
    21: "HUE",
    23: "JAE",
    29: "MAL",
    41: "SEV"
}

# Map the values
df_triaje["province"] = df_triaje["province"].map(map_province)

# Check the mapping
print("Value counts of 'province' column after conversion:")
df_triaje["province"].value_counts(dropna=False)


## resource_dest_province_id

In [ ]:
# Original value counts
print("Value counts of 'resource_dest_province_id' column before conversion:")
df_triaje["resource_dest_province_id"].value_counts(dropna=False)


## IDALERTANTE - alert_receiver

This variable provides us with an identification code for the entity that received the alert. We have a total of 23 different IDs (as we will verify below). According to clinical indications, they will be regrouped into the following 6 groups:

- GROUP 1 '112': '112'.
- GROUP 2 'USER': 'USER' (User).
- GROUP 3 'POL/FG' (Police/Fire Brigade): '91', '92', '62', '80', '091', '092', '062', '080'.
- GROUP 4 'HS' (Health System): 'APS', 'SEU', 'SNU', 'HOS', '61', '061', 'ECA', 'PAC', 'DCC'.
- GROUP 5 'TELE' (Teleassistance): 'SAT'.
- GROUP 6 'OTHERS': 'PCI', 'OTR', 'SR', 'CRE', 'FAX', 'TEL', 'INA', 'RES'.

In [ ]:
# Original value counts
print("Value counts of 'alert_receiver' column before conversion:")
df_triaje["alert_receiver"].value_counts(dropna=False)

In [ ]:
# Map raw alert receiver codes to grouped categories.
def map_alert_receiver(x):
    """
    This function:
    - Normalizes input values (strip spaces, uppercase).
    - Handles None/NaN safely.
    - Uses a dictionary-based mapping for maintainability.
    - Returns np.nan when the code is unknown or invalid.
    """

    # If null → return NaN
    if pd.isna(x):
        return np.nan

    # Normalize input (string only)
    x = str(x).strip().upper()
    if x == "":
        return np.nan

    # Dictionary of mappings
    mapping_dict = {
        # Group 1
        "112": "112",

        # Group 2
        "USU": "USER",

        # Group 3: POL/FG
        "91": "POL/FG", "92": "POL/FG", "62": "POL/FG", "80": "POL/FG",
        "091": "POL/FG", "092": "POL/FG", "062": "POL/FG", "080": "POL/FG",

        # Group 4: HS
        "APS": "HS", "SEU": "HS", "SNU": "HS", "HOS": "HS",
        "61": "HS", "061": "HS", "ECA": "HS", "PAC": "HS", "DCC": "HS",

        # Group 5: TELE
        "SAT": "TELE",

        # Group 6: OTHERS
        "PCI": "OTHERS", "OTR": "OTHERS", "SR": "OTHERS", "CRE": "OTHERS",
        "FAX": "OTHERS", "TEL": "OTHERS", "INA": "OTHERS", "RES": "OTHERS",
    }

    # Return mapped value or NaN if not found
    return mapping_dict.get(x, np.nan)


# Map the values
df_triaje["alert_receiver"] = df_triaje["alert_receiver"].apply(map_alert_receiver)

# Check the mapping
print("Value counts of 'alert_receiver' column after conversion:")
df_triaje["alert_receiver"].value_counts(dropna=False)

## IDESCENARIO - location_patient
- GROUP 1 'HO' (Home): 'DO'
- GROUP 2 'PR' (Public Road): 'VP'
- GROUP 3 'OTHERS': 'PI', 'CA', 'OT'.

In [ ]:
# Original value counts
print("Value counts of 'location_patient' column before conversion:")
df_triaje["location_patient"].value_counts(dropna=False)

In [ ]:
# Map dictionary
map_location = {
    "DO": "HO",
    "VP": "PR",
    "CA": "OTHERS",
    "OT": "OTHERS",
    "PI": "OTHERS"
}

# Map the values
df_triaje["location_patient"] = df_triaje["location_patient"].map(map_location)

# Check the mapping
print("Value counts of 'location_patient' column after conversion:")
df_triaje["location_patient"].value_counts(dropna=False)

## IDTIPORECURSO - resource_type

In [ ]:
# Original value counts
print("Value counts of 'resource_type' column before conversion:")
df_triaje["resource_type"].value_counts(dropna=False)



In [ ]:
print(f"\nNumber of unique values in resource_type: {df_triaje['resource_type'].nunique()}")

## IDTARJETACORAZON - healthcard

In [ ]:
df_triaje["healthcard"].value_counts(dropna = False).head(20)

In [ ]:
print(f"Number of None values in 'healthcard' is: {df_triaje['healthcard'].isna().sum()}")
df_triaje["healthcard"] = df_triaje["healthcard"].notna().astype(int)
df_triaje["healthcard"].dtype

In [ ]:
df_triaje["healthcard"].value_counts(dropna = False)

## IDTARJETAAIRE - aircard

In [ ]:
df_triaje["aircard"].value_counts(dropna = False)

In [ ]:
print(f"Number of None values in 'aircard' is: {df_triaje['aircard'].isna().sum()}")
df_triaje["aircard"] = df_triaje["aircard"].notna().astype(int)
df_triaje["aircard"].dtype

In [ ]:
df_triaje["aircard"].value_counts(dropna = False)

## ESVULNERABLE - is_vulnerable

In [ ]:
df_triaje["is_vulnerable"].value_counts(dropna = False)

In [ ]:
df_triaje["is_vulnerable"] = df_triaje["is_vulnerable"].notna().astype(int)
df_triaje["is_vulnerable"].dtype

In [ ]:
df_triaje["is_vulnerable"].value_counts(dropna = False)

## IDHCDM

In [ ]:
df_triaje["hcdm_id"].isna().sum()

In [ ]:
df_triaje["hcdm_id"].notna().sum()

## Tests

In [ ]:
df_triaje.info()

## Transform data types

In [ ]:
df_triaje = dp.transform_column_dtypes(df_triaje)

In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(100)

# Joining tables

## triage questions

In [ ]:
# Load table
df_preguntas = pq.read_table(os.path.join(settings.source_tables_path, 
                                          "preguntas_triaje/preguntas_respuesta_detalle_20260122.parquet")).to_pandas()
df_preguntas.shape

In [ ]:
df_preguntas.info()

In [ ]:
df_preguntas.head(6)

In [ ]:
# Test value counts
df_preguntas["Q7"].value_counts(dropna = False)

In [ ]:
# Fill missing values with "0"
questions_cols = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7"]
df_preguntas.loc[:, questions_cols] = df_preguntas.loc[:, questions_cols].fillna("0")



In [ ]:
# Initial count of total rows and unique IDs
print(f"Rows in df_triaje before joining df_preguntas: {len(df_triaje)}")
print(f"Rows in df_preguntas is: {len(df_preguntas)}")
print(f"\nNumber of unique DEMANDAPK in df_preguntas before deduplication: {df_preguntas['DEMANDAPK'].nunique()}")

# Calculate the number of duplicate rows in df_preguntas based on DEMANDAPK
n_duplicated = df_preguntas["DEMANDAPK"].duplicated().sum()
print(f"Number of duplicated DEMANDAPK in df_preguntas is: {n_duplicated}")

# Filter df_preguntas to include only IDs present in df_triaje to avoid fan-out
# and remove duplicates from DEMANDAPK to prevent row multiplication during merge
df_preguntas_unique = df_preguntas[df_preguntas["DEMANDAPK"].isin(df_triaje["demandpk"])]
df_preguntas_unique = df_preguntas_unique.drop_duplicates(subset=["DEMANDAPK"])

# Count unique IDs after filtering and deduplication process
print(f"Number of unique DEMANDAPK in df_preguntas after deduplication: {df_preguntas_unique['DEMANDAPK'].nunique()}")

# Left join df_triaje with the deduplicated questions dataframe
df_triaje = df_triaje.merge(
    df_preguntas_unique[["DEMANDAPK"] + questions_cols],
    how="left",
    left_on="demandpk",
    right_on="DEMANDAPK"
)

# Drop the redundant join key
df_triaje = df_triaje.drop(columns=["DEMANDAPK"])

# Fill missing values in question columns with "0"
df_triaje.loc[:, questions_cols] = df_triaje.loc[:, questions_cols].fillna("0")

# Rename the question columns to lowercase
df_triaje = df_triaje.rename(
    columns={col: col.lower() for col in questions_cols}
)

# Summarize overall numbers
dp.summarize_numbers(df_triaje, key_col, id_columns)

In [ ]:
df_triaje.head(5)

In [ ]:
dp.summarize_numbers(df_triaje, key_col, id_columns)

df_triaje.info()

## Transform data types

In [ ]:
df_triaje = dp.transform_column_dtypes(df_triaje)
df_triaje.info()

# Creation of new variables

## Juicio clínico - icd

In [ ]:
cols_icd = ["icd1", "icd2", "icd3"]

df_triaje[cols_icd] = (
    df_triaje[cols_icd]
    .apply(lambda col: col.astype("string").str.strip())     # strip whitespace
    .replace(r'^\s*$', pd.NA, regex=True)                    # empty strings → NA
    .replace(["nan", "NaN", "None", "none"], pd.NA)          # textual nulls → NA
    .astype("string")                                        # final enforcement
)


In [ ]:
df_triaje.head(10)

In [ ]:
def combine_clean_columns(df, columns, sep):
    """
    Clean ICD-related columns and combine valid entries into a single string.
    Missing values are returned as pd.NA.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.
    columns : list
        List of ICD columns to clean and combine.
    sep : str
        Separator used to join non-missing cleaned values.
    """

    def clean_value(x):
        """Normalize different missing representations to None."""
        if pd.isna(x):  # Detects real NaN/None/NaT
            return None
        if isinstance(x, str):
            x_strip = x.strip()
            # Remove empty strings and stringified missing markers
            if x_strip.lower() in ["", "nan", "none"]:
                return None
            return x_strip
        return x

    # Apply cleaning column-wise (avoid deprecated applymap)
    cleaned = df[columns].apply(lambda col: col.apply(clean_value))

    # Combine valid entries using the provided separator
    combined = cleaned.apply(
        lambda row: sep.join(str(x) for x in row if x is not None),
        axis=1
    )

    # Convert empty strings to real missing values
    combined = combined.replace("", pd.NA)

    return combined

df_triaje["icd_emerg_combined"] = combine_clean_columns(
    df_triaje,
    columns=["icd1", "icd2", "icd3"],
    sep=" | "       # You choose the separator manually
)

# Transform dtype and reorder dataframe
df_triaje = dp.transform_column_dtypes(df_triaje)


In [ ]:
df_triaje.head(10)

In [ ]:
def analyze_icd_missing_by_demand_type(df):
    """
    Analyzes missing values (pd.NA) in 'icd_emerg_combined' grouped by 'demand_type_1'.
    Provides absolute and relative values with human-readable descriptions of demand types.
    """
    
    # 1. Calculate totals per demand type to get the base for percentages
    demand_totals = df.groupby('demand_type_1').size().reset_index(name='total_cases')

    # 2. Calculate missing values in 'icd_emerg_combined' per demand type
    missing_data = df.groupby('demand_type_1').agg(
        missing_icd_count=('icd_emerg_combined', lambda x: x.isna().sum())
    ).reset_index()

    # 3. Merge both metrics
    report = pd.merge(missing_data, demand_totals, on='demand_type_1')

    # 4. Calculate the percentage of missing values within each demand category
    report['%_missing_icd'] = (report['missing_icd_count'] / report['total_cases']) * 100

    # 5. Mapping demand codes to human-readable descriptions (Integrated map)
    description_map = {
        16: "Dyspnea / respiratory distress",
        23: "Non-traumatic chest pain",
        36: "Unconsciousness / cardiac arrest",
        54: "Stroke / cerebrovascular accident",
        58: "Unconsciousness / cardiac arrest"
    }
    
    report['demand_description'] = report['demand_type_1'].map(description_map).fillna("Other / Unknown")

    # 6. Final formatting and ordering
    # We order by percentage to see which pathology is "losing" more information
    ordered_cols = [
        'demand_type_1', 'demand_description', 
        'missing_icd_count', 'total_cases', '%_missing_icd'
    ]
    
    report = report[ordered_cols].sort_values(by='%_missing_icd', ascending=False)
    report = report.round(2)

    return report

# Execution
missing_icd_report = analyze_icd_missing_by_demand_type(df_triaje)
print("=== Missing ICD-10 Analysis by Demand Type ===")
print(missing_icd_report)

In [ ]:
df_triaje_icd = df_triaje[df_triaje["icd_emerg_combined"].str.contains("47.9", na=False)]



In [ ]:
df_triaje_icd.head(10)

In [ ]:
df_triaje['icd_emerg_combined'].isna().sum()

In [ ]:
dp.summarize_numbers(df_triaje, key_col, id_columns)

In [ ]:
df_triaje.loc[
    df_triaje["icd_emerg_combined"].str.contains("798", na=False),
    "resolution_code_id"
].value_counts(dropna=False)


In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(10)

## P1 Predicted

In [ ]:
print("Value counts of 'p1_predicted' column before conversion:")
df_triaje['p1_predicted'].value_counts(dropna = False)

In [ ]:
print(df_triaje['p1_predicted'].dtype)

In [ ]:
# p1_predicted is the priority predicted during the call, assigned by the system
# 1 is maximum priority, 0 is non-maximum priority
df_triaje['p1_predicted'] = np.where(df_triaje['p1_predicted'] == '1', '1', '0')

In [ ]:
print("Value counts of 'p1_predicted' column after conversion:")
df_triaje['p1_predicted'].value_counts(dropna = False)

In [ ]:
print(df_triaje['p1_predicted'].dtype)

## P1 Real

In [ ]:
# Load table linking ICD to P1, together with special cases containing specific triage questions
icd_table = pd.read_csv(
    os.path.join(settings.source_tables_path, "icd/icd_p1outcomes_20251218.tsv"),
    dtype=str,
    encoding="utf-8",
    sep="\t"
)


In [ ]:
icd_table.head(10)

In [ ]:
icd_table["AND/Exclusion"].value_counts(dropna=False)


In [ ]:
icd_table.info()

In [ ]:
print(len(icd_table))
icd_table["ICD_original"].nunique()

In [ ]:
import pandas as pd
import numpy as np

def process_icd_codes(df, icd_combined_col, reference_table, reference_icd_col):
    """
    Process combined ICD codes to detect ICD exclusions, P1 priority flags,
    and apply additional specific triage rules based on AND/Exclusion categories.

    Parameters
    ----------
    df : pandas.DataFrame
        Main dataset containing the combined ICD codes column.
    icd_combined_col : str
        Name of the column containing combined ICD codes separated by "|".
    reference_table : pandas.DataFrame
        ICD reference table containing ICD codes, P1 labels, and Exclusion labels.
    reference_icd_col : str
        Name of the ICD column inside the reference table (e.g., 'ICD_original').

    Returns
    -------
    df_filtered : pandas.DataFrame
        Filtered dataframe with a new column 'p1_real_emerg'.
    n_rows_excluded_icd : int
        Number of rows removed due to exclusion criteria.
    """

    # ===============================================================
    # 1. Normalize ICD reference table and create lookup dictionaries
    # ===============================================================
    ref = reference_table.copy()

    ref[reference_icd_col] = ref[reference_icd_col].astype("string").str.strip()

    ref["p1_real_emerg"] = ref["P1 outcomes"].apply(
        lambda x: 1 if isinstance(x, str) and x.strip().upper() == "P1" else 0
    )
    ref["exclusion_flag"] = ref["AND/Exclusion"].apply(
        lambda x: 1 if isinstance(x, str) and x.strip().lower() == "exclusion" else 0
    )

    p1_map = dict(zip(ref[reference_icd_col], ref["p1_real_emerg"]))
    exclusion_map = dict(zip(ref[reference_icd_col], ref["exclusion_flag"]))

    # ===============================================================
    # 2. Function to evaluate a single row of combined ICD codes
    # ===============================================================
    def evaluate_icd_row(text):
        if pd.isna(text):
            # Maintain None for rows without ICD information
            return {"exclude": False, "p1": None}

        raw_codes = [c.strip() for c in str(text).split("|")]
        codes = [c for c in raw_codes if c != ""]

        for code in codes:
            if code in exclusion_map and exclusion_map[code] == 1:
                return {"exclude": True, "p1": None}

        for code in codes:
            if code in p1_map and p1_map[code] == 1:
                return {"exclude": False, "p1": 1}

        return {"exclude": False, "p1": 0}


    # ===============================================================
    # 3. Apply evaluation row-wise across the dataset
    # ===============================================================
    evaluation = df[icd_combined_col].apply(evaluate_icd_row)

    df["p1_real_emerg"] = evaluation.apply(lambda r: r["p1"])
    df["exclude_row"]   = evaluation.apply(lambda r: r["exclude"])

    n_rows_excluded_icd = df["exclude_row"].sum()

    # ===============================================================
    # 4. Remove excluded rows
    # ===============================================================
    n_rows_previous = len(df)
    
    df_filtered = df[df["exclude_row"] == False].drop(columns=["exclude_row"]).copy()
    df_filtered = df_filtered.reset_index(drop=True)

    n_rows_final = len(df_filtered)

    # ===============================================================
    # 5. Apply specific triage rules based on AND/Exclusion categories
    # ===============================================================

    # Identify ICD groups associated with special rules
    icd_speech = ref.loc[
        ref["AND/Exclusion"] == "Habla con dificultad", 
        reference_icd_col
    ].dropna().astype(str).tolist()

    icd_normal = ref.loc[
        ref["AND/Exclusion"] == "Vida Normal y/o limitada, pero se vale por si mismo",
        reference_icd_col
    ].dropna().astype(str).tolist()

    # Split ICDs into lists per row
    df_filtered["icd_list"] = (
        df_filtered[icd_combined_col]
        .astype("string")
        .fillna("<NA>")
        .apply(lambda x: [c.strip() for c in x.split("|")] if x != "<NA>" else [])
    )

    # Helper function for matching
    def row_matches_any(row_icds, target_icds):
        return any(c in target_icds for c in row_icds)

    # Define common masks for rule application
    p1_before = df_filtered["p1_real_emerg"].copy()
    mask_has_icd = df_filtered["p1_real_emerg"].notna()

    # --- Rule set A: "Habla con dificultad"
    mask_speech_icd = df_filtered["icd_list"].apply(lambda lst: row_matches_any(lst, icd_speech))

    mask_speech_true = (
        ((df_filtered["demand_type_1"] == 16) & (df_filtered["q3"] == "b")) |
        ((df_filtered["demand_type_1"] == 54) & (df_filtered["q3"] == "d"))
    )

    # Apply rule A ensuring NA values are preserved
    df_filtered.loc[mask_speech_icd & mask_speech_true & mask_has_icd, "p1_real_emerg"] = 1
    df_filtered.loc[mask_speech_icd & ~mask_speech_true & mask_has_icd, "p1_real_emerg"] = 0


    # --- Rule set B: "Vida Normal y/o limitada, pero se vale por si mismo"
    mask_normal_icd = df_filtered["icd_list"].apply(lambda lst: row_matches_any(lst, icd_normal))

    mask_normal_true = (
        (df_filtered["demand_type_1"] == 54) & (df_filtered["q6"].isin(["a", "0"]))
    )

    # Apply rule B ensuring NA values are preserved
    df_filtered.loc[mask_normal_icd & mask_normal_true & mask_has_icd, "p1_real_emerg"] = 1
    df_filtered.loc[mask_normal_icd & ~mask_normal_true & mask_has_icd, "p1_real_emerg"] = 0

    # Count how many rows were changed by the specific rules
    n_p1_updated_specific_rules = (df_filtered["p1_real_emerg"] != p1_before).sum()

    # Remove helper column
    df_filtered = df_filtered.drop(columns=["icd_list"])

    # ===============================================================
    # 6. Return dataframe and statistics
    # ===============================================================
    stats = {
        "n_rows_previous": n_rows_previous,
        "n_rows_final": n_rows_final,
        "n_rows_excluded_icd": n_rows_excluded_icd,
        "n_p1_updated_specific_rules": n_p1_updated_specific_rules,
    }

    # ===============================================================
    # 7. Print summary and finalize data types
    # ===============================================================
    print("\n================ ICD Processing Summary ================\n")
    print(f"✔ Total rows before applying ICD exclusion criteria: {stats['n_rows_previous']}")
    print(f"✔ Total rows removed due to ICD exclusion rules: {stats['n_rows_excluded_icd']}")
    print(f"✔ Total rows remaining after applying ICD exclusion criteria: {stats['n_rows_final']}")
    print(f"✔ Rows where P1 was reassigned due to specific triage rules: {stats['n_p1_updated_specific_rules']}")
    print(f"✔ Rows with <NA> in p1_real_emerg (no ICD data): {df_filtered['p1_real_emerg'].isna().sum()}")
    print("\n========================================================\n")

    # Final conversion to nullable Int8 for memory efficiency and NA support
    df_filtered["p1_real_emerg"] = df_filtered["p1_real_emerg"].astype("Int8")

    return df_filtered.reset_index(drop=True), n_rows_excluded_icd

# Call the function
df_triaje, n_rows_excluded_icd = process_icd_codes(
    df=df_triaje,
    icd_combined_col="icd_emerg_combined",
    reference_table=icd_table,
    reference_icd_col="ICD_original"
)

In [ ]:
df_triaje["p1_real_emerg"].value_counts(dropna=False)

In [ ]:
import pandas as pd

def analyze_p1_missing_by_demand_type(df):
    """
    Analyzes missing values (pd.NA), positive cases (1) and negative cases (0) 
    in 'p1_real_emerg' grouped by 'demand_type_1'. Provides absolute and relative values.
    """
    
    # 1. Calculate totals per demand type
    demand_totals = df.groupby('demand_type_1').size().reset_index(name='total_cases')

    # 2. Calculate missing values, P1 (1) and non-P1 (0) counts
    # We use lambda functions to handle the nullable Int8/Int64 types correctly
    metrics_data = df.groupby('demand_type_1').agg(
        p1_count=('p1_real_emerg', lambda x: (x == 1).sum()),
        p0_count=('p1_real_emerg', lambda x: (x == 0).sum()),
        missing_p1_count=('p1_real_emerg', lambda x: x.isna().sum())
    ).reset_index()

    # 3. Merge metrics
    report = pd.merge(metrics_data, demand_totals, on='demand_type_1')

    # 4. Calculate the percentage of missing target values
    report['%_missing_p1'] = (report['missing_p1_count'] / report['total_cases']) * 100

    # 5. Mapping demand codes to human-readable descriptions
    description_map = {
        16: "Dyspnea / respiratory distress",
        23: "Non-traumatic chest pain",
        36: "Unconsciousness / cardiac arrest",
        54: "Stroke / cerebrovascular accident",
        58: "Unconsciousness / cardiac arrest"
    }
    
    report['demand_description'] = report['demand_type_1'].map(description_map).fillna("Other / Unknown")

    # 6. Final formatting and ordering
    # Columns ordered for a clear view of Data Quality and Class Balance
    ordered_cols = [
        'demand_type_1', 'demand_description', 
        'p1_count', 'p0_count', 'missing_p1_count', 
        'total_cases', '%_missing_p1'
    ]
    
    report = report[ordered_cols].sort_values(by='%_missing_p1', ascending=False)
    report = report.round(2)

    return report

# Execution example
target_analysis_report = analyze_p1_missing_by_demand_type(df_triaje)
print("=== Target (p1_real_emerg) Analysis by Demand Type ===")
print(target_analysis_report)

In [ ]:
import pandas as pd

def update_p1_by_death_resolution(
        df, 
        target_col,
        icd_col="icd_emerg_combined",
        resolution_col="resolution_code_id",
        death_codes=None
    ):
    """
    Assigns target_col = 1 for cases where the ICD is missing AND 
    the resolution code indicates a death outcome.
    """
    
    # -----------------------------------------------------------
    # 1. Default death resolution code sets
    # -----------------------------------------------------------
    if death_codes is None:
        death_codes = ["32", "33"]

    # -----------------------------------------------------------
    # 2. Identify missing ICDs and extract resolution codes
    # -----------------------------------------------------------
    # We look for rows that currently have <NA> in the target column
    icd_missing_mask = df[target_col].isna()

    # Process resolution codes (handling potential '|' separators)
    res_expanded = (
        df[resolution_col]
        .astype("string")
        .str.split("|")
        .explode()
        .str.strip()
    )

    # -----------------------------------------------------------
    # 3. Identify rows with death codes
    # -----------------------------------------------------------
    has_death = res_expanded.isin(death_codes).groupby(level=0).any()

    # Ensure alignment with original dataframe
    has_death = has_death.reindex(df.index, fill_value=False)

    # -----------------------------------------------------------
    # 4. Apply rule: target IS <NA> AND death → target = 1
    # -----------------------------------------------------------
    rule_mask = icd_missing_mask & has_death
    
    # Calculation of stats before the update to avoid NA comparison issues
    n_updated = rule_mask.sum()

    # Update only the missing values that meet the criteria
    df.loc[rule_mask, target_col] = 1

    print("\n================ Death Resolution Rules Summary ================\n")
    print(f"✔ Rows updated from <NA> to P1 (Death criteria): {n_updated}")
    print("================================================================\n")

    # Ensure final type is nullable Int8
    df[target_col] = df[target_col].astype("Int8")

    return df, n_updated

# Execution
df_triaje, n_updated_death = update_p1_by_death_resolution(
    df_triaje, 
    target_col="p1_real_emerg"
)

In [ ]:
df_triaje["p1_real_emerg"].value_counts(dropna=False)

In [ ]:
# Execution example
target_analysis_report = analyze_p1_missing_by_demand_type(df_triaje)
print("=== Target (p1_real_emerg) Analysis by Demand Type ===")
print(target_analysis_report)

In [ ]:
# Assign p1_real_bps = 1 for patients who died or were transferred to the hospital

def assign_p1_based_on_resolution(
        df, 
        icd_col="icd_emerg_combined",
        resolution_col="resolution_code_id",
        p1_col="p1_real_bps",
        death_codes=None,
        transfer_codes=None
    ):
    """
    Assign p1_real_bps = 1 for specific cases:
    - icd_col is missing (any missing type)
    - AND resolution_code_id contains codes related to death or hospital transfer

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.
    icd_col : str
        Column containing combined ICD codes.
    resolution_col : str
        Column containing resolution codes (may include '|' separators).
    p1_col : str
        Name of the P1 flag column to modify.
    death_codes : list or None
        List of resolution codes indicating patient death.
    transfer_codes : list or None
        List of resolution codes indicating hospital transfer.

    Returns
    -------
    df : pandas.DataFrame
        Updated dataframe with modified p1_real_bps.
    n_p1_updated : int
        Number of rows where p1_real_bps was set to 1 by this rule.
    """
    # Initiate column equal to previous p1_real_emerg
    df[p1_col] = df["p1_real_emerg"]
    
    # -----------------------------------------------------------
    # Default resolution code sets (safe initialization)
    # -----------------------------------------------------------
    if death_codes is None:
        death_codes = ["32", "33"]

    if transfer_codes is None:
        transfer_codes = [
            "12", "18A", "18B", "23", "24", "26", "26B",
            "35", "35B", "36", "36B", "36C", "38A", "38B",
            "43", "44", "45"
        ]

    # -----------------------------------------------------------
    # Identify missing ICDs (any type of missing)
    # -----------------------------------------------------------
    icd_missing_mask = df[icd_col].isna() | (
        df[icd_col].astype(str).str.strip().isin(["", "nan", "None"])
    )

    # -----------------------------------------------------------
    # Explode resolution codes safely into a MultiIndex Series
    # -----------------------------------------------------------
    resolution_expanded = (
        df[resolution_col]
        .astype("string")
        .str.split("|")
        .explode()
        .str.strip()
        .replace({"": None, "nan": None, "None": None})
    )

    # -----------------------------------------------------------
    # Identify rows having ANY death or transfer code
    # -----------------------------------------------------------
    has_death = resolution_expanded.isin(death_codes).groupby(level=0).any()
    has_transfer = resolution_expanded.isin(transfer_codes).groupby(level=0).any()

    # Guarantee alignment with the original DataFrame
    has_death = has_death.reindex(df.index, fill_value=False)
    has_transfer = has_transfer.reindex(df.index, fill_value=False)

    # -----------------------------------------------------------
    # Apply rule: ICD missing AND (death OR transfer) → p1_real = 1
    # -----------------------------------------------------------
    rule_mask = icd_missing_mask & (has_death | has_transfer)

    n_p1_updated = rule_mask.sum()

    df.loc[rule_mask, p1_col] = 1
    
    return df, n_p1_updated

df_triaje, n_updated = assign_p1_based_on_resolution(df_triaje)

print("Rows updated to P1 because of death/transfer:", n_updated)



In [ ]:
df_triaje["p1_real_bps"].value_counts(dropna=False)

In [ ]:
df_triaje.info()

In [ ]:
df_triaje[df_triaje["icd_emerg_combined"].notna()].head(5)

## triage

In [ ]:
# Create the 'triage' variable:
# - triage = 0 → all demand-specific question fields (q1–q7) are equal to "0"
#                meaning no demand-specific information was collected during the call.
# - triage = 1 → at least one demand-specific question has a value different from "0",
#                indicating that structured triage information was captured.


question_cols = ["q1","q2","q3","q4","q5","q6","q7"]

# Compute triage flag: 1 if any values != "0", otherwise 0
df_triaje["triage"] = (
    (df_triaje[question_cols] != "0").any(axis=1).astype("int8")
)

# Convert to categorical dtype
df_triaje["triage"] = df_triaje["triage"].astype("category")


## Date/Time variables

In [ ]:
# Create the day of the week (number, Monday=0, Sunday=6) variable
df_triaje['day_week'] = df_triaje['demand_date'].dt.dayofweek

In [ ]:
df_triaje['day_week'].value_counts(dropna = False)

In [ ]:
# Create the month (number) variable
df_triaje['month'] = df_triaje['demand_date'].dt.month

In [ ]:
df_triaje['month'].value_counts(dropna = False)

In [ ]:
# Create the year variable
df_triaje['year'] = df_triaje['demand_date'].dt.year

In [ ]:
df_triaje['year'].value_counts(dropna = False)

In [ ]:
# Display the new columns and the original date column
print(df_triaje[['demand_date', 'day_week', 'month', 'year']].head())

In [ ]:
def add_season_variable(df, date_col, output_col="season"):
    """
    Add a 'season' numerical column to the dataframe based on a date column.
    Seasons are encoded as integers to preserve ordinality and ensure 
    compatibility with ML models (avoiding categorical mismatch errors).
    
    Mapping:
        - 0: Spring (21/03–20/06)
        - 1: Summer (21/06–22/09)
        - 2: Autumn (23/09–20/12)
        - 3: Winter (rest of the dates)

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    date_col : str
        Name of the datetime column used to compute the season.
    output_col : str, default="season"
        Name of the resulting season column.
    """

    # Compute MMDD as a single integer (e.g., March 21 -> 321)
    # Using float temporarily to handle potential NaNs safely
    md = (df[date_col].dt.month * 100 + df[date_col].dt.day)

    # Define seasonal conditions based on Northern Hemisphere cutoff dates
    conditions = [
        md.between(321, 620, inclusive="both"),   # Spring
        md.between(621, 922, inclusive="both"),   # Summer
        md.between(923, 1220, inclusive="both")   # Autumn
    ]
    
    # Define numerical choices (0 to 2). Winter will be the default (3).
    choices = [0, 1, 2]

    # Assign seasonal values using fast NumPy selection
    # Default is 3 (Winter)
    df[output_col] = np.select(conditions, choices, default=3)

    # Convert to int8 for memory efficiency and ensure compatibility
    # If the date is NaT, the result of md is NaN, and we should preserve it as -1
    df[output_col] = df[output_col].astype(np.int8)
    df.loc[df[date_col].isna(), output_col] = -1

    return df

# Usage
df_triaje = add_season_variable(df_triaje, "demand_date", output_col="season")

In [ ]:
df_triaje["season"].value_counts(dropna=False)

In [ ]:

def add_time_slot(df, primary_date_col, backup_date_col, output_col="time_of_day"):
    """
    Create a numerical time-of-day slot from a primary datetime column, 
    falling back to a backup column if the primary is NaT. Rows where both 
    columns are NaT are excluded.

    Mapping (Hours 0–23):
        - 0: early_morning (00–05)
        - 1: morning       (06–11)
        - 2: afternoon     (12–17)
        - 3: night         (18–23)
    """
    initial_rows = len(df)

    # 1. Create a combined hour series (Coalesce logic)
    # If primary is NaT, use backup. If both are NaT, it remains NaN.
    combined_hour = df[primary_date_col].dt.hour.fillna(df[backup_date_col].dt.hour)

    # 2. Drop rows where both date columns are NaT
    df = df.dropna(subset=[primary_date_col, backup_date_col], how='all').copy()
    
    final_rows = len(df)
    n_rows_excluded_time = initial_rows - final_rows

    # 3. Apply numerical binning to the hours of the remaining rows
    # We re-calculate combined_hour for the filtered dataframe
    hour_filtered = df[primary_date_col].dt.hour.fillna(df[backup_date_col].dt.hour)
    
    df[output_col] = pd.cut(
        hour_filtered,
        bins=[-1, 5, 11, 17, 23],
        labels=[0, 1, 2, 3],
        right=True
    ).astype(np.int8)

    # 4. Reporting
    print(f"=== Time Slot Processing ===")
    print(f"Original rows: {initial_rows}")
    print(f"Excluded rows (both dates NaT): {n_rows_excluded_time}")
    print(f"Final rows: {final_rows}")
    print(f"============================\n")

    return df, n_rows_excluded_time

# Usage
df_triaje, n_rows_excluded_time = add_time_slot(
    df_triaje, 
    primary_date_col="contact_entry_time", 
    backup_date_col="activation_time", 
    output_col="time_of_day"
)

In [ ]:
df_triaje["time_of_day"].value_counts(dropna=False)

## Transform data types

In [ ]:
df_triaje = dp.transform_column_dtypes(df_triaje)

In [ ]:
df_triaje.info()

In [ ]:
import pandas as pd

# 1. Filtrar las filas donde resolution_code_id es igual a 31
df_triaje_test = df_triaje[df_triaje['resolution_code_id'] == 31]

# 2. Agrupar por 'demand_type' y contar las ocurrencias
resultado = df_triaje_test.groupby('demand_type_1').size().reset_index(name='cantidad_filas')

print(resultado)

In [ ]:
df_triaje.sample(20)

# Final exclusion criteria

## Resolucion code: found death (31)

In [ ]:
# EDA: exploration of data

def analyze_resolution_codes_31(df):
    """
    Analyzes resolution codes within the triage dataset, specifically focusing on code '31'
    and tracking missing ICD information across different demand types.
    """
    
    # Calculate original totals per demand type
    totals_per_type = df.groupby('demand_type_1').size().reset_index(name='total_original')

    # Filter rows containing '31' and create a subset for detailed analysis
    df_res_31 = df[df['resolution_code_id'].astype(str).str.contains('31', na=False)].copy()

    # Calculate metrics for count of '31' and missing ICD entries
    # Counts total rows for '31' and rows where icd_emerg_combined is null
    analysis_results = df_res_31.groupby('demand_type_1').agg(
        count_31=('demand_type_1', 'size'),
        icd_missing_in_31=('icd_emerg_combined', lambda x: x.isna().sum())
    ).reset_index()

    # Merge with original totals and calculate percentages
    final_report = pd.merge(analysis_results, totals_per_type, on='demand_type_1')

    # Percentage of '31' cases over the original total for that demand type
    final_report['%_31_total'] = (final_report['count_31'] / final_report['total_original']) * 100

    # Percentage of missing ICD entries specifically within the '31' group
    final_report['%_icd_missing_in_31'] = (final_report['icd_missing_in_31'] / final_report['count_31']) * 100

    # Map technical codes to human-readable descriptions
    description_map = {
        16: "Dyspnea / respiratory distress",
        23: "Non-traumatic chest pain",
        36: "Unconsciousness / cardiac arrest",
        54: "Stroke / cerebrovascular accident",
        58: "Unconsciousness / cardiac arrest"
    }
    final_report['description'] = final_report['demand_type_1'].map(description_map)

    # Final formatting: reorder columns and round numerical values to 2 decimals
    ordered_cols = [
        'demand_type_1', 'description', 'count_31', 'total_original', 
        '%_31_total', 'icd_missing_in_31', '%_icd_missing_in_31'
    ]
    
    final_report = final_report[ordered_cols].sort_values(by='description')
    final_report = final_report.round(2)

    return final_report

# Call the function
resolution_analysis_df = analyze_resolution_codes_31(df_triaje)
print(resolution_analysis_df)

In [ ]:
# Exclusion of rows with missing ICD information in resolution code '31' (found death)

def exclude_missing_icd_in_code31(df):
    """
    Excludes rows that have resolution code '31' (handling pipe-separated values)
    and missing ICD information, with robust null handling.
    """
    def contains_code_31(val):
        # pd.isna handles None, NaN, and <NA>
        if pd.isna(val):
            return False
        
        # Convert to string and handle potential stringified nulls
        val_str = str(val).strip().lower()
        if val_str in ['none', 'nan', '']:
            return False
            
        # Split by '|', strip whitespace from each element, and check for '31'
        codes = [code.strip() for code in val_str.split('|')]
        return '31' in codes

    # Identify the rows to be excluded
    # We use .isna() for icd_emerg_combined as it is the standard for missing data
    mask_to_exclude = (
        df['resolution_code_id'].apply(contains_code_31) & 
        df['icd_emerg_combined'].isna()
    )
    
    # Calculate counts for reporting
    initial_rows = len(df)
    n_rows_excluded_code31 = mask_to_exclude.sum()
    
    # Perform exclusion
    df_cleaned = df[~mask_to_exclude].copy()
    final_rows = len(df_cleaned)
    
    # Print results for audit trail
    print(f"Rows before exclusion: {initial_rows:,}")
    print(f"Rows excluded (Exact Code 31 with missing ICD): {n_rows_excluded_code31:,}")
    print(f"Rows after exclusion: {final_rows:,}")
    
    return df_cleaned, n_rows_excluded_code31

# Execution:
df_triaje, n_rows_excluded_code31 = exclude_missing_icd_in_code31(df_triaje)

# Data preparation for modeling

## triaje answers - one-hot encoding

In [ ]:

def encode_triage_questions(df, question_cols):
    """
    Performs custom One-Hot Encoding for triage questions with robust dtype handling.
    Values: 1 (selected), 0 (not selected), -1 (question not asked).
    """
    df_encoded = df.copy()
    
    for col in question_cols:
        # Identify rows where the question was NOT asked (original value "0")
        not_asked_mask = df_encoded[col].astype(str) == "0"
        
        # Get standard One-Hot Encoding. 
        # dtype=int prevents the boolean-to-integer FutureWarning.
        dummies = pd.get_dummies(df_encoded[col], prefix=col, dtype=int)
        
        # Drop the '0' column as it represents the "not asked" state which we handle with -1
        col_zero = f"{col}_0"
        if col_zero in dummies.columns:
            dummies = dummies.drop(columns=[col_zero])
        
        # Now we can safely assign -1 because the dtype is already integer
        dummies.loc[not_asked_mask, :] = -1
        
        # Add to the main dataframe and remove original column
        df_encoded = pd.concat([df_encoded, dummies], axis=1)
        df_encoded = df_encoded.drop(columns=[col])
        
    return df_encoded

# Execute encoding
df_triaje = encode_triage_questions(df_triaje, question_cols)

## Free text - SBERT embedding

In [ ]:

def get_text_embeddings(df, column_name='literal_reason', batch_size=512):
    """
    Generate semantic text embeddings for medical descriptions using SBERT.
    The resulting dataframe includes only 'demandpk', the original text column, 
    and the new embeddings.
    
    This function saves the result to a Parquet file if settings.run_embedding is enabled.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe containing 'demandpk' and the text column.
    column_name : str, default='literal_reason'
        Name of the column to encode.
    batch_size : int, default=128
        Number of sentences processed at once (uses GPU if available).

    Returns
    -------
    pd.DataFrame or None
        A filtered dataframe containing [demandpk, column_name, emb_0...emb_n]
        if the process is enabled, otherwise returns None.
    """
    
    # Check the run_embedding setting to handle specific cases
    if settings.run_embedding == 0:
        print("--- Skipping embedding generation as run_embedding is set to 0 ---")
        return None
    elif settings.run_embedding == 1:
        # Proceed with embedding generation
        pass
    else:
        # Print error if the value is neither 0 nor 1
        print("Error: settings.run_embedding must be either 0 or 1.")
        return None
    
    print(f"--- Generating embeddings for {len(df)} rows in column '{column_name}' ---")
    
    # Load a lightweight yet powerful multilingual model
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    # Comprehensive handling of nulls, None, and empty strings
    clean_sentences = (
        df[column_name]
        .replace(["", None, "None"], np.nan)
        .fillna("Sin descripción")
        .astype(str)
        .tolist()
    )
    
    # Generate embeddings (Vectorization)
    embeddings = model.encode(
        clean_sentences, 
        batch_size=batch_size, 
        show_progress_bar=True, 
        convert_to_numpy=True
    )
    
    # Create descriptive column names for the embedding dimensions
    embedding_colnames = [f"emb_{i}" for i in range(embeddings.shape[1])]
    
    # Construct the embeddings dataframe preserving the original index
    embeddings_df = pd.DataFrame(embeddings, columns=embedding_colnames, index=df.index)
    
    # Concatenate only specific columns: demandpk, literal_reason and all embeddings
    id_cols = [c for c in ['demandpk', column_name] if c in df.columns]
    result_df = pd.concat([df[id_cols], embeddings_df], axis=1)
    
    print(f"Successfully generated {embeddings.shape[1]}-dimensional embeddings.")

    # Export to Parquet as we know run_embedding is enabled
    output_path = os.path.join(settings.source_tables_path, settings.table_embedding_path)
    result_df.to_parquet(output_path, index=False)
    
    return result_df

In [ ]:
# Call the function
df_embedding = get_text_embeddings(df_triaje)

In [ ]:
if 'df_embedding' in locals() and df_embedding is not None:
    df_embedding.info(verbose=True, show_counts=True)

In [ ]:
if 'df_embedding' in locals() and df_embedding is not None:
    df_embedding.head()

In [ ]:
# Merge embedding columns with main table
# --- Handling Embedding Table (New or Saved) ---

if settings.run_embedding == 0:
    # Scenario: Embeddings already exist, just load them
    print("--- Loading previously saved embedding table ---")
    df_embedding = pq.read_table(
        os.path.join(settings.source_tables_path, settings.table_embedding_path)
    ).to_pandas()
else:
    # Scenario: df_embedding was just created by the function get_text_embeddings()
    print("--- Using newly created embeddings ---")

# --- Unified Merge Logic ---

# We only perform the join if df_embedding is available (either loaded or created)
if df_embedding is not None:
    print(f"""--- Merging embeddings ({len(df_embedding)} rows) 
    with main table ({len(df_triaje)} rows)---""")
    
    # We drop 'literal_reason' from the embedding table to avoid duplication
    # We use 'on' because both tables must share the 'demandpk' column
    df_triaje = df_triaje.merge(
        df_embedding.drop(columns=['literal_reason'], errors='ignore'), 
        on='demandpk', 
        how='left'
    )
    
    # Optional: ensure memory efficiency by deleting the temporary dataframe
    del df_embedding
    print("--- Merge completed successfully ---")
else:
    print("Error: No embedding data found to merge.")

In [ ]:
df_triaje.info(verbose=True, show_counts=True)

In [ ]:
df_triaje.head()

## Transform data types

In [ ]:
df_triaje = dp.transform_column_dtypes(df_triaje)

In [ ]:
df_triaje.info(max_cols=500)

# Consort diagram

In [ ]:
# Print excluded rows consecutively
print(f"Rows from original dataset: {n_rows_source_data:,}")
print(f"\nRows excluded with duplicated demandpk: {n_rows_excluded_demandpk:,}")
print(f"Rows excluded in age column: {n_rows_excluded_age:,}")
print(f"Rows excluded with missing values in sex: {n_rows_excluded_sex:,}")
print(f"Rows excluded with trauma diagnoses in ICD: {n_rows_excluded_icd:,}")
print(f"Rows excluded with no specific time: {n_rows_excluded_time:,}")
print(f"Rows excluded without ICD codes in code resolution 31 (found death): {n_rows_excluded_code31:,}")

# Calculate total excluded rows 
n_rows_excluded_total = (
    n_rows_excluded_demandpk + 
    n_rows_excluded_age + 
    n_rows_excluded_sex + 
    n_rows_excluded_icd + 
    n_rows_excluded_time +
    n_rows_excluded_code31
)

print(f"\nRows from original dataset: {n_rows_source_data:,}")
print(f"Total rows excluded: {n_rows_excluded_total:,}")
print(f"% rows excluded: {((n_rows_excluded_total/n_rows_source_data)*100).round(2):,}")
print(f"Final rows in df_triaje: {len(df_triaje):,}\n")

# Validate exclusion logic
if len(df_triaje) == (n_rows_source_data - n_rows_excluded_total):
    print("✅ THE LOGIC FOR EXCLUSION CRITERIA IS OK")
else:
    diff = len(df_triaje) - (n_rows_source_data - n_rows_excluded_total)
    print(f"❌ THERE IS AN ERROR CALCULATING EXCLUSION CRITERIA. Difference: {diff} rows")

# Summarize final numbers
dp.summarize_numbers(df_triaje, key_col, id_columns)

# Data Export

In [ ]:
# Save the cleaned dataframe to a Parquet file
df_triaje.to_parquet(os.path.join(settings.source_tables_path, settings.triaje_table_cleaned_path), index=False)

# Define the path and check if the file was created correctly
export_path = os.path.join(settings.source_tables_path, settings.triaje_table_cleaned_path)
if os.path.exists(export_path):
    file_size_mb = os.path.getsize(export_path) / (1024 * 1024)
    print(f"✅ Export successful!")
    print(f"--- Path: {export_path}")
    print(f"--- Size: {file_size_mb:.2f} MB")
else:
    print(f"❌ Export failed: File not found at {export_path}")

In [ ]:
import psutil
import torch
import os

def check_resources():
    """
    Checks and prints the available CPU RAM and GPU VRAM resources.
    Useful for determining the optimal batch_size for embedding generation.
    """
    print("=== System Resource Report ===")
    
    # 1. CPU RAM Monitoring
    vm = psutil.virtual_memory()
    total_ram = vm.total / (1024**3)
    available_ram = vm.available / (1024**3)
    used_ram_pct = vm.percent
    
    print(f"\n[CPU RAM]")
    print(f"  Total:     {total_ram:.2f} GB")
    print(f"  Available: {available_ram:.2f} GB")
    print(f"  Usage:     {used_ram_pct}%")

    # 2. GPU VRAM Monitoring (NVIDIA)
    print(f"\n[GPU / CUDA]")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            total_vram = props.total_memory / (1024**3)
            # Memory currently allocated by PyTorch
            allocated_vram = torch.cuda.memory_allocated(i) / (1024**3)
            # Memory reserved by PyTorch but not necessarily used
            reserved_vram = torch.cuda.memory_reserved(i) / (1024**3)
            # Rough estimate of free VRAM
            free_vram = total_vram - reserved_vram
            
            print(f"  Device {i}: {props.name}")
            print(f"    Total VRAM:     {total_vram:.2f} GB")
            print(f"    Allocated VRAM: {allocated_vram:.2f} GB")
            print(f"    Free VRAM (est): {free_vram:.2f} GB")
    else:
        print("  CUDA is not available. Using CPU only.")

    # 3. Process Specific Usage
    process = psutil.Process(os.getpid())
    process_ram = process.memory_info().rss / (1024**3)
    print(f"\n[Current Jupyter Process]")
    print(f"  RAM Usage: {process_ram:.2f} GB")

if __name__ == "__main__":
    check_resources()

# EDA

## Tests

In [ ]:
df_triaje.info(verbose=True, show_counts=True)

In [ ]:
df_triaje.head(5)

In [ ]:
target_analysis_report = analyze_p1_missing_by_demand_type(df_triaje)
print("=== Target (p1_real_emerg) Missing Analysis by Demand Type ===")
print(target_analysis_report)

In [ ]:
df_triaje['cadrec'].value_counts(dropna = False)

In [ ]:
icd_cols = ["icd1", "icd2", "icd3"]

df_filtered = dp.filter_rows_by_icd(
    df=df_triaje,
    icd_value="047.9",
    icd_columns=icd_cols
)

df_filtered.head()

## Resolution codes

In [ ]:
import re

# -------------------------------------------------------
# 1. Split combined codes (e.g. "34 | 28") into one code per row
# -------------------------------------------------------

# Split by '|' and clean whitespace
df_codes = (
    df_triaje['resolution_code_id']
    .astype(str)
    .str.split('|')
    .apply(lambda x: [item.strip() for item in x])  # remove spaces
    .explode()                                      # create one row per code
    .replace({'': None, 'nan': None, 'None': None}) # normalize missing values
)

# -------------------------------------------------------
# 2. Helper function to sort by numeric + alphabetical part
# -------------------------------------------------------
def split_code(code):
    """
    Splits a resolution code into numeric and alphabetical parts.
    Example: "34A" -> (34, "A"), "36" -> (36, "")
    """
    if code is None:
        return (float('inf'), "")  # send None to the bottom
    
    code = str(code).strip()
    
    # Capture digits + optional letters
    match = re.match(r"(\d+)([A-Za-z]*)", code)
    if match:
        num = int(match.group(1))
        let = match.group(2)
        return (num, let)
    else:
        # If unexpected format, send it to the bottom
        return (float('inf'), code)

# -------------------------------------------------------
# 3. Unique values sorted
# -------------------------------------------------------
unique_codes = df_codes.dropna().unique()
sorted_codes = sorted(unique_codes, key=split_code)

# -------------------------------------------------------
# 4. Value counts sorted according to numeric+letter order
# -------------------------------------------------------
vc = df_codes.value_counts(dropna=False)

# Create sorted index
sorted_index = sorted(vc.index, key=split_code)

# Reindex to apply sorted order
vc_sorted = vc.reindex(sorted_index)

# -------------------------------------------------------
# 5. Display results
# -------------------------------------------------------
vc_sorted


# DATA QUALITY

## Analysis of Missing

In [ ]:
dp.analyze_missing_values(df_triaje)